# Stage A: Official AVEC E-DAIC Split Reconstruction

Reconstruct the official AVEC 2019 participant-level E-DAIC split from the existing processed multimodal utterances.

Target: **163 train / 56 development / 56 test participants**.

No participant may occur in more than one split. Existing processed are not overwritten.

In [1]:
from pathlib import Path
import pickle
import re
import numpy as np
import pandas as pd

In [ ]:
# Paths
PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OFFICIAL_DIR = PROCESSED_DIR / "official_split"
OFFICIAL_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Raw:", RAW_DIR)
print("Processed:", PROCESSED_DIR)
print("Output:", OFFICIAL_DIR)

In [ ]:
if not RAW_DIR.exists():
    raise FileNotFoundError(f"Raw directory does not exist: {RAW_DIR}")
print("Files in data/raw:")
for p in sorted(RAW_DIR.iterdir()):
    print(" -", p.name)

In [ ]:
def find_file_by_stem(directory, stem):
    candidates=[p for p in directory.iterdir() if p.is_file() and p.stem.lower()==stem.lower()]
    if not candidates:
        raise FileNotFoundError(f"Could not find '{stem}' in {directory}")
    if len(candidates)>1:
        raise RuntimeError("Multiple files found:"+"".join(map(str,candidates)))
    return candidates[0]

def load_table(path):
    if path.suffix.lower()=='.csv': return pd.read_csv(path)
    if path.suffix.lower() in {'.xlsx','.xls'}: return pd.read_excel(path)
    raise ValueError(f"Unsupported file type: {path.suffix}")

TRAIN_SPLIT_PATH=find_file_by_stem(RAW_DIR,'train_split')
DEV_SPLIT_PATH=find_file_by_stem(RAW_DIR,'dev_split')
TEST_SPLIT_PATH=find_file_by_stem(RAW_DIR,'test_split')
print(TRAIN_SPLIT_PATH); print(DEV_SPLIT_PATH); print(TEST_SPLIT_PATH)

In [9]:
train_split_df=load_table(TRAIN_SPLIT_PATH)
dev_split_df=load_table(DEV_SPLIT_PATH)
test_split_df=load_table(TEST_SPLIT_PATH)
print(train_split_df.shape, dev_split_df.shape, test_split_df.shape)
print("Train columns:", list(train_split_df.columns))
print("Dev columns:", list(dev_split_df.columns))
print("Test columns:", list(test_split_df.columns))

(163, 6) (56, 6) (56, 6)
Train columns: ['Participant_ID', 'Gender', 'PHQ_Binary', 'PHQ_Score', 'PCL-C (PTSD)', 'PTSD Severity']
Dev columns: ['Participant_ID', 'Gender', 'PHQ_Binary', 'PHQ_Score', 'PCL-C (PTSD)', 'PTSD Severity']
Test columns: ['Participant_ID', 'Gender', 'PHQ_Binary', 'PHQ_Score', 'PCL-C (PTSD)', 'PTSD Severity']


In [7]:
PARTICIPANT_COLUMN_CANDIDATES=["Participant_ID","Participant ID","participant_id","participant","Participant","participantid","ParticipantID"]
def find_participant_column(df):
    for c in PARTICIPANT_COLUMN_CANDIDATES:
        if c in df.columns: return c
    norm={str(c).strip().lower().replace(' ','_'):c for c in df.columns}
    for c in PARTICIPANT_COLUMN_CANDIDATES:
        if c.lower().replace(' ','_') in norm: return norm[c.lower().replace(' ','_')]
    raise KeyError(f"Participant-ID column not found. Columns: {list(df.columns)}")

def normalize_participant_id(value):
    if pd.isna(value): return None
    value=str(value).strip()
    if re.fullmatch(r"\d+\.0",value): value=value[:-2]
    return value

TRAIN_PARTICIPANT_COL=find_participant_column(train_split_df)
DEV_PARTICIPANT_COL=find_participant_column(dev_split_df)
TEST_PARTICIPANT_COL=find_participant_column(test_split_df)
print(TRAIN_PARTICIPANT_COL, DEV_PARTICIPANT_COL, TEST_PARTICIPANT_COL)

Participant_ID Participant_ID Participant_ID


In [8]:
train_participants={normalize_participant_id(x) for x in train_split_df[TRAIN_PARTICIPANT_COL] if normalize_participant_id(x) is not None}
dev_participants={normalize_participant_id(x) for x in dev_split_df[DEV_PARTICIPANT_COL] if normalize_participant_id(x) is not None}
test_participants={normalize_participant_id(x) for x in test_split_df[TEST_PARTICIPANT_COL] if normalize_participant_id(x) is not None}
all_official_participants=train_participants|dev_participants|test_participants
print("Train:",len(train_participants),"Dev:",len(dev_participants),"Test:",len(test_participants),"Total:",len(all_official_participants))

Train: 163 Dev: 56 Test: 56 Total: 275


In [10]:
EXPECTED_TRAIN,EXPECTED_DEV,EXPECTED_TEST,EXPECTED_TOTAL=163,56,56,275
assert len(train_participants)==EXPECTED_TRAIN, f"Train expected 163, got {len(train_participants)}"
assert len(dev_participants)==EXPECTED_DEV, f"Dev expected 56, got {len(dev_participants)}"
assert len(test_participants)==EXPECTED_TEST, f"Test expected 56, got {len(test_participants)}"
assert len(all_official_participants)==EXPECTED_TOTAL, f"Total expected 275, got {len(all_official_participants)}"
print('✓ Official participant counts verified.')

✓ Official participant counts verified.


In [11]:
train_dev_overlap=train_participants&dev_participants
train_test_overlap=train_participants&test_participants
dev_test_overlap=dev_participants&test_participants
print(train_dev_overlap, train_test_overlap, dev_test_overlap)
assert not train_dev_overlap and not train_test_overlap and not dev_test_overlap
print('✓ No participant appears in more than one split.')

set() set() set()
✓ No participant appears in more than one split.


In [12]:
split_summary=pd.DataFrame({'split':['train','development','test'],'participants':[len(train_participants),len(dev_participants),len(test_participants)]})
display(split_summary)

,split,participants
0,train,163
1,development,56
2,test,56


# Load the existing processed utterances

Those files already encode the previous split. Each processed record is assigned according to its participant.

In [13]:
FULL_DATASET_PATH=PROCESSED_DIR/'hubertdeberta_dataset.pkl'
if not FULL_DATASET_PATH.exists():
    raise FileNotFoundError(f"Full dataset not found: {FULL_DATASET_PATH}")
with open(FULL_DATASET_PATH,'rb') as f:
    processed_samples=pickle.load(f)
print('Processed utterances:',len(processed_samples))
print('Type:',type(processed_samples))

Processed utterances: 23436
Type: <class 'list'>


In [14]:
sample=processed_samples[0]
print('Keys:',sample.keys())
print('Participant:',sample.get('participant_id'))
print('Utterance:',sample.get('utterance_id'))
print('Audio shape:',np.asarray(sample['audio_embedding']).shape)
print('Text shape:',np.asarray(sample['text_embedding']).shape)

Keys: dict_keys(['participant_id', 'utterance_id', 'text', 'confidence', 'duration', 'audio_embedding', 'text_embedding'])
Participant: 300
Utterance: 300_0
Audio shape: (768,)
Text shape: (768,)


In [15]:
missing=[i for i,s in enumerate(processed_samples) if s.get('participant_id') is None]
assert not missing, f"Records missing participant_id: {len(missing)}"
processed_participant_ids={normalize_participant_id(s['participant_id']) for s in processed_samples}
print('Processed participants:',len(processed_participant_ids))
print('Official participants:',len(all_official_participants))

Processed participants: 275
Official participants: 275


In [16]:
processed_not_in_official=processed_participant_ids-all_official_participants
official_not_in_processed=all_official_participants-processed_participant_ids
print('Processed not official:',len(processed_not_in_official), sorted(processed_not_in_official)[:20])
print('Official not processed:',len(official_not_in_processed), sorted(official_not_in_processed)[:20])
assert not processed_not_in_official
assert not official_not_in_processed
print('✓ Participant coverage matches exactly.')

Processed not official: 0 []
Official not processed: 0 []
✓ Participant coverage matches exactly.


In [17]:
participant_to_split={}
for p in train_participants: participant_to_split[p]='train'
for p in dev_participants: participant_to_split[p]='dev'
for p in test_participants: participant_to_split[p]='test'
assert len(participant_to_split)==275

In [18]:
official_train=[]; official_dev=[]; official_test=[]; unassigned_samples=[]
for sample in processed_samples:
    p=normalize_participant_id(sample['participant_id'])
    split=participant_to_split.get(p)
    if split=='train': official_train.append(sample)
    elif split=='dev': official_dev.append(sample)
    elif split=='test': official_test.append(sample)
    else: unassigned_samples.append(sample)
print('Train segments:',len(official_train))
print('Dev segments:',len(official_dev))
print('Test segments:',len(official_test))
print('Unassigned:',len(unassigned_samples))
assert not unassigned_samples
assert len(official_train)+len(official_dev)+len(official_test)==len(processed_samples)

Train segments: 13451
Dev segments: 4904
Test segments: 5081
Unassigned: 0


In [19]:
def participant_set(samples):
    return {normalize_participant_id(s['participant_id']) for s in samples}

rt=participant_set(official_train); rd=participant_set(official_dev); rx=participant_set(official_test)
assert rt==train_participants and rd==dev_participants and rx==test_participants
assert not(rt&rd) and not(rt&rx) and not(rd&rx)
print('✓ Reconstructed participant sets exactly match official files.')

✓ Reconstructed participant sets exactly match official files.


In [20]:
split_segment_summary=pd.DataFrame({'split':['train','development','test'],'participants':[len(rt),len(rd),len(rx)],'segments':[len(official_train),len(official_dev),len(official_test)]})
display(split_segment_summary)

,split,participants,segments
0,train,163,13451
1,development,56,4904
2,test,56,5081


# Add PHQ-8 labels

The classification rule is PHQ-8 ≥ 10 → depressed (1), PHQ-8 < 10 → control (0). The continuous PHQ-8 total is retained for the later participant-level severity analysis.

In [ ]:
PHQ_PATH=find_file_by_stem(RAW_DIR,'Detailed_PHQ8_Labels')
phq_df=load_table(PHQ_PATH)
print('PHQ file:',PHQ_PATH)
print('Shape:',phq_df.shape)
print('Columns:',list(phq_df.columns))
display(phq_df.head())

In [22]:
PHQ_PARTICIPANT_COL=find_participant_column(phq_df)
PHQ_COLUMN_CANDIDATES=['PHQ_8Total','PHQ8_Total','PHQ_8_Total','PHQ8','PHQ-8','PHQ_Score','PHQ Score']
def find_phq_column(df):
    for c in PHQ_COLUMN_CANDIDATES:
        if c in df.columns: return c
    norm={str(c).strip().lower().replace(' ','_'):c for c in df.columns}
    for c in PHQ_COLUMN_CANDIDATES:
        if c.lower().replace(' ','_') in norm: return norm[c.lower().replace(' ','_')]
    raise KeyError(f'PHQ-8 total column not found. Columns: {list(df.columns)}')
PHQ_TOTAL_COL=find_phq_column(phq_df)
print('Participant column:',PHQ_PARTICIPANT_COL)
print('PHQ total column:',PHQ_TOTAL_COL)

Participant column: Participant_ID
PHQ total column: PHQ_8Total


In [24]:
# CHECK PHQ-8 COVERAGE

missing_phq = (
    all_official_participants
    - set(phq_lookup.keys())
)

print(
    "Official participants without PHQ-8:",
    len(missing_phq)
)

if missing_phq:
    print(
        "Participants without PHQ-8 labels:"
    )
    print(
        sorted(missing_phq)
    )

print()
print(
    "PHQ-8 coverage:",
    f"{len(all_official_participants) - len(missing_phq)} "
    f"/ {len(all_official_participants)}"
)

Official participants without PHQ-8: 56
Participants without PHQ-8 labels:
['600', '602', '604', '605', '606', '607', '609', '615', '618', '619', '620', '622', '623', '624', '625', '626', '629', '631', '634', '635', '636', '637', '638', '640', '649', '650', '651', '652', '655', '656', '658', '659', '661', '663', '664', '666', '669', '676', '679', '682', '683', '688', '689', '691', '693', '696', '699', '705', '708', '709', '710', '712', '715', '716', '717', '718']

PHQ-8 coverage: 219 / 275


In [25]:
# CHECK PHQ-8 COVERAGE BY OFFICIAL SPLIT

def check_phq_coverage(participants, split_name):
    
    available = participants & set(phq_lookup.keys())
    missing = participants - set(phq_lookup.keys())
    
    print(f"{split_name}:")
    print(f"  Total participants: {len(participants)}")
    print(f"  PHQ-8 available:    {len(available)}")
    print(f"  PHQ-8 missing:      {len(missing)}")
    
    if missing:
        print(f"  Missing IDs:        {sorted(missing)}")
    
    print()


check_phq_coverage(
    train_participants,
    "TRAIN",
)

check_phq_coverage(
    dev_participants,
    "DEVELOPMENT",
)

check_phq_coverage(
    test_participants,
    "TEST",
)

TRAIN:
  Total participants: 163
  PHQ-8 available:    163
  PHQ-8 missing:      0

DEVELOPMENT:
  Total participants: 56
  PHQ-8 available:    56
  PHQ-8 missing:      0

TEST:
  Total participants: 56
  PHQ-8 available:    0
  PHQ-8 missing:      56
  Missing IDs:        ['600', '602', '604', '605', '606', '607', '609', '615', '618', '619', '620', '622', '623', '624', '625', '626', '629', '631', '634', '635', '636', '637', '638', '640', '649', '650', '651', '652', '655', '656', '658', '659', '661', '663', '664', '666', '669', '676', '679', '682', '683', '688', '689', '691', '693', '696', '699', '705', '708', '709', '710', '712', '715', '716', '717', '718']



In [26]:
print("TRAIN SPLIT")
display(train_split_df.head())
print(train_split_df.columns.tolist())

print("\nDEV SPLIT")
display(dev_split_df.head())
print(dev_split_df.columns.tolist())

print("\nTEST SPLIT")
display(test_split_df.head())
print(test_split_df.columns.tolist())

TRAIN SPLIT


,Participant_ID,Gender,PHQ_Binary,PHQ_Score,PCL-C (PTSD),PTSD Severity
0,302,male,0,4,0,28
1,303,female,0,0,0,17
2,304,female,0,6,0,20
3,305,male,0,7,0,28
4,307,female,0,4,0,23


['Participant_ID', 'Gender', 'PHQ_Binary', 'PHQ_Score', 'PCL-C (PTSD)', 'PTSD Severity']

DEV SPLIT


,Participant_ID,Gender,PHQ_Binary,PHQ_Score,PCL-C (PTSD),PTSD Severity
0,300,male,0,2,0,25
1,301,male,0,3,0,17
2,306,female,0,0,0,21
3,317,male,0,8,1,51
4,320,female,0,11,1,64


['Participant_ID', 'Gender', 'PHQ_Binary', 'PHQ_Score', 'PCL-C (PTSD)', 'PTSD Severity']

TEST SPLIT


,Participant_ID,Gender,PHQ_Binary,PHQ_Score,PCL-C (PTSD),PTSD Severity
0,600,female,0,5,0,23.0
1,602,female,1,13,1,67.0
2,604,male,1,12,0,30.0
3,605,male,0,2,0,23.0
4,606,female,0,5,0,46.0


['Participant_ID', 'Gender', 'PHQ_Binary', 'PHQ_Score', 'PCL-C (PTSD)', 'PTSD Severity']


In [27]:
train_meta = train_split_df.copy()
dev_meta = dev_split_df.copy()
test_meta = test_split_df.copy()

train_meta["split"] = "train"
dev_meta["split"] = "dev"
test_meta["split"] = "test"

official_metadata = pd.concat(
    [train_meta, dev_meta, test_meta],
    ignore_index=True,
)

official_metadata["participant_id"] = (
    official_metadata["Participant_ID"]
    .apply(normalize_participant_id)
)

official_metadata["phq8_total"] = pd.to_numeric(
    official_metadata["PHQ_Score"],
    errors="coerce",
)

official_metadata["label"] = (
    official_metadata["PHQ_Binary"]
    .astype(int)
)

display(official_metadata.head())

,Participant_ID,Gender,PHQ_Binary,PHQ_Score,PCL-C (PTSD),PTSD Severity,split,participant_id,phq8_total,label
0,302,male,0,4,0,28.0,train,302,4,0
1,303,female,0,0,0,17.0,train,303,0,0
2,304,female,0,6,0,20.0,train,304,6,0
3,305,male,0,7,0,28.0,train,305,7,0
4,307,female,0,4,0,23.0,train,307,4,0


In [32]:
# Verify official labels
official_metadata["score_threshold_label"] = (
    official_metadata["phq8_total"] >= 10
).astype(int)

mismatch_df = official_metadata[
    official_metadata["label"]
    != official_metadata["score_threshold_label"]
].copy()

print(
    "PHQ_Binary vs PHQ_Score >= 10 mismatches:",
    len(mismatch_df)
)

print("\nOfficial label distribution:")
print(
    official_metadata.groupby(["split", "label"])
    .size()
    .unstack(fill_value=0)
)

PHQ_Binary vs PHQ_Score >= 10 mismatches: 20

Official label distribution:
label    0   1
split         
dev     44  12
test    39  17
train  126  37


In [33]:
print(
    official_metadata.groupby("split")["participant_id"]
    .nunique()
)

split
dev       56
test      56
train    163
Name: participant_id, dtype: int64


In [34]:
# Build the participant metadata lookup
participant_metadata = (
    official_metadata
    .set_index("participant_id")
    .to_dict("index")
)

print(
    "Participants in metadata:",
    len(participant_metadata)
)

assert len(participant_metadata) == 275

Participants in metadata: 275


In [35]:
# Add labels to the utterances 
def add_official_metadata(samples):

    output = []

    for sample in samples:

        participant_id = normalize_participant_id(
            sample["participant_id"]
        )

        if participant_id not in participant_metadata:
            raise KeyError(
                f"Participant {participant_id} "
                "not found in official metadata."
            )

        meta = participant_metadata[participant_id]

        sample_copy = dict(sample)

        sample_copy["split"] = meta["split"]
        sample_copy["phq8_total"] = float(
            meta["phq8_total"]
        )
        sample_copy["label"] = int(
            meta["label"]
        )

        output.append(sample_copy)

    return output

In [36]:
official_train = add_official_metadata(
    official_train
)

official_dev = add_official_metadata(
    official_dev
)

official_test = add_official_metadata(
    official_test
)

# Save the rebuilt official split

In [ ]:
TRAIN_OUTPUT=OFFICIAL_DIR/'hubertdeberta_train.pkl'
DEV_OUTPUT=OFFICIAL_DIR/'hubertdeberta_dev.pkl'
TEST_OUTPUT=OFFICIAL_DIR/'hubertdeberta_test.pkl'
for path,data in [(TRAIN_OUTPUT,official_train),(DEV_OUTPUT,official_dev),(TEST_OUTPUT,official_test)]:
    with open(path,'wb') as f: pickle.dump(data,f)
    print('Saved:',path)

In [ ]:
SUMMARY_OUTPUT = OFFICIAL_DIR / 'split_summary.csv'
MANIFEST_OUTPUT = OFFICIAL_DIR / 'participant_manifest.csv'

# 1. Build participant-level manifest

participant_manifest = official_metadata[
    [
        'participant_id',
        'split',
        'PHQ_Score',
        'PHQ_Binary'
    ]
].copy()

participant_manifest = participant_manifest.rename(
    columns={
        'PHQ_Score': 'phq8_total',
        'PHQ_Binary': 'label'
    }
)

participant_manifest['participant_id'] = (
    participant_manifest['participant_id'].astype(int)
)

participant_manifest['label'] = (
    participant_manifest['label'].astype(int)
)

participant_manifest['phq8_total'] = (
    pd.to_numeric(
        participant_manifest['phq8_total'],
        errors='coerce'
    )
)

# Sort consistently
participant_manifest = participant_manifest.sort_values(
    ['split', 'participant_id']
).reset_index(drop=True)

# 2. Validate participant coverage

assert (
    participant_manifest['participant_id'].nunique()
    == len(all_official_participants)
), "Participant manifest does not contain all official participants."

assert (
    participant_manifest['participant_id'].nunique()
    == 275
), "Expected exactly 275 unique participants."

assert (
    participant_manifest['label'].notna().all()
), "Some participants are missing PHQ_Binary labels."

assert (
    participant_manifest['split'].notna().all()
), "Some participants are missing split assignments."

# 3. Build split summary

final_summary = (
    participant_manifest
    .groupby(['split', 'label'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

final_summary = final_summary.rename(
    columns={
        0: 'n_label_0',
        1: 'n_label_1'
    }
)

# Add total participants per split
final_summary['n_participants'] = (
    final_summary[['n_label_0', 'n_label_1']]
    .sum(axis=1)
)

# Put columns in a clean order
final_summary = final_summary[
    [
        'split',
        'n_participants',
        'n_label_0',
        'n_label_1'
    ]
]

# 4. Save outputs

participant_manifest.to_csv(
    MANIFEST_OUTPUT,
    index=False
)

final_summary.to_csv(
    SUMMARY_OUTPUT,
    index=False
)

print('Saved:', SUMMARY_OUTPUT)
print('Saved:', MANIFEST_OUTPUT)

print('\nSplit summary:')
display(final_summary)

print('\nParticipant manifest:')
display(participant_manifest.head(10))

# Stage A Complete

Expected participant allocation: **163 train / 56 development / 56 test / 275 total**.